# Code Ownership, notebook 3: the encoders and the feature sets

Frozen encoders (no weights are changed), mean-pooled and L2-normalised: UniXcoder-base and CodeBERT-base (the proposal's choices) and all-mpnet-base-v2 (a general text
encoder as a control). Feature sets A-D (answer; + similarities; + answer-times-reference; joint question+answer) with and without the hand-built features.

In [1]:
import sys, warnings, json, pickle
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../../helpers").resolve()))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
import ko_data as K, ko_models as M
from core import metrics, group_bootstrap, paired_diff_bootstrap, qwk, band
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 90)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
d = K.load()
dev = d[d.role == "dev"].reset_index(drop=True)
test = d[d.role == "test"].reset_index(drop=True)
y, groups, folds = dev.score.to_numpy(), dev.snippet_id.to_numpy(), dev.fold.to_numpy()
def ci(p, stat):
    return group_bootstrap(y, p, groups, stat, n=1000)
def sp(a, b):
    return spearmanr(a, b).correlation
def mae(a, b):
    return float(np.mean(np.abs(a - b)))
def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

In [2]:
import time
from embedder import Encoder
available = {}
for key in ("unixcoder", "codebert", "mpnet"):
    local = Encoder._local(key)
    available[key] = local or "hub cache"
print(available)
ENCODERS = [k for k in ("unixcoder", "codebert", "mpnet") if (Encoder._local(k) or k == "mpnet")]
print("encoders used:", ENCODERS)

{'unixcoder': 'C:\\Users\\sadee\\OneDrive\\Documents\\SLIIT\\Forth Year\\Research\\Project\\J26-SE-363\\backend\\agentic_framework\\app\\agents\\performance_assessment\\factor_models\\07_code_ownership\\training\\dataset\\cache\\models\\unixcoder-base', 'codebert': 'C:\\Users\\sadee\\OneDrive\\Documents\\SLIIT\\Forth Year\\Research\\Project\\J26-SE-363\\backend\\agentic_framework\\app\\agents\\performance_assessment\\factor_models\\07_code_ownership\\training\\dataset\\cache\\models\\codebert-base', 'mpnet': 'hub cache'}
encoders used: ['unixcoder', 'codebert', 'mpnet']


In [3]:
H = K.hand_features(dev)
info = []
EMB = {}
for key in ENCODERS:
    t = time.time(); E = K.embeddings(key, dev); EMB[key] = E
    sets = K.feature_sets(E, H)
    info.append({"encoder": key, "embedding dim": E["ans"].shape[1], **{f"dim {s}": v.shape[1] for s, v in sets.items()}, "seconds": round(time.time() - t, 1)})
pd.DataFrame(info).set_index("encoder")

,embedding dim,dim A,dim B,dim C,dim D,dim A+H,dim B+H,dim C+H,dim D+H,seconds
encoder,,,,,,,,,,
unixcoder,768,768,771,1539,771,780,783,1551,783,0.1
codebert,768,768,771,1539,771,780,783,1551,783,0.1
mpnet,768,768,771,1539,771,780,783,1551,783,0.1


A first look: cosine similarity between the answer and the reference explanation, by written level. If the embeddings carried the 'is it right' signal this would climb with the level.

In [4]:
rows = []
for key, E in EMB.items():
    for part in ("ref", "code", "ques"):
        c = (E["ans"] * E[part]).sum(1)
        rows.append({"encoder": key, "answer vs": part, "Spearman with level": round(sp(c, dev.rating), 3), **{f"level {r}": round(float(c[dev.rating == r].mean()), 3) for r in range(1, 6)}})
pd.DataFrame(rows)

,encoder,answer vs,Spearman with level,level 1,level 2,level 3,level 4,level 5
0,unixcoder,ref,0.147,0.368,0.479,0.407,0.432,0.458
1,unixcoder,code,0.163,0.420,0.468,0.470,0.504,0.487
2,unixcoder,ques,0.240,0.332,0.380,0.421,0.421,0.418
3,codebert,ref,0.161,0.958,0.971,0.960,0.965,0.967
4,codebert,code,0.142,0.875,0.884,0.869,0.881,0.891
5,codebert,ques,0.150,0.968,0.973,0.973,0.974,0.972
6,mpnet,ref,0.110,0.454,0.597,0.499,0.508,0.554
7,mpnet,code,0.193,0.441,0.528,0.446,0.501,0.537
8,mpnet,ques,0.100,0.516,0.573,0.543,0.539,0.568


## Reading the results

**Encoders.** UniXcoder-base, CodeBERT-base (the proposal's choices) and all-mpnet-base-v2 (a general text control), frozen, mean-pooled, L2-normalised; 768 dimensions each. CodeT5+ was not used (its loader runs downloaded code).

**The raw cosine similarities carry almost no signal.** Spearman with the written level of the cosine between the answer and the reference is 0.11 to 0.16, and to the question 0.10 to 0.24. CodeBERT's cosines sit at 0.87 to 0.97 for every level (the unfine-tuned mean-pooled space is crowded), so the similarity features are nearly constant; whatever the encoders add comes from the answer embedding itself, which the learned head reads.
